In [13]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [14]:
# Load a small, fast embedding model
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded!")
print(f"Model produces {model.get_sentence_embedding_dimension()} dimensional embeddings")

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000164D097D090>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: d7f5f026-ebb8-44c2-9273-c8960494b55c)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Loading embedding model...


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000164D097F610>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: d2d71021-9ff0-434b-9830-72b01ff2173b)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000164D097F890>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 7c3b016c-3dbf-4af8-aae5-873432bc7a22)')' thrown while requesting HEAD https://huggingface.

✅ Model loaded!
Model produces 384 dimensional embeddings


In [15]:
# Simple example
text = "The cat sat on the mat"

# Generate embedding
embedding = model.encode(text)

print(f"Original text: {text}")
print(f"Embedding shape: {embedding.shape}")
print(f"Embedding type: {type(embedding)}")
print(f"\nFirst 10 values: {embedding[:10]}")

Original text: The cat sat on the mat
Embedding shape: (384,)
Embedding type: <class 'numpy.ndarray'>

First 10 values: [ 0.13040186 -0.01187012 -0.02811704  0.05123863 -0.05597441  0.03019154
  0.03016129  0.02469839 -0.01837056  0.05876678]


In [16]:
def cosine_similarity(vec1, vec2):
    """
    Calculate cosine similarity between two vectors.
    
    Returns a score between -1 and 1 (higher = more similar)
    """
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)

print("✅ Similarity function ready!")

✅ Similarity function ready!


In [17]:
# Create test sentences
sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",      # Similar meaning, different words
    "Dogs are loyal animals",          # Different topic
    "Python is a programming language" # Completely unrelated
]

# Generate embeddings for all sentences
embeddings = model.encode(sentences)

# Compare first sentence to all others
print("Comparing to: 'The cat sat on the mat'\n")
for i, sentence in enumerate(sentences):
    similarity = cosine_similarity(embeddings[0], embeddings[i])
    print(f"Similarity to '{sentence}'")
    print(f"Score: {similarity:.3f}\n")

Comparing to: 'The cat sat on the mat'

Similarity to 'The cat sat on the mat'
Score: 1.000

Similarity to 'A feline rested on the rug'
Score: 0.564

Similarity to 'Dogs are loyal animals'
Score: 0.165

Similarity to 'Python is a programming language'
Score: 0.031



In [18]:
# Sample knowledge base
documents = [
    "Python is a high-level programming language known for simplicity",
    "Machine learning enables computers to learn from data",
    "Neural networks are inspired by biological brains",
    "Dogs are loyal and friendly pets that need exercise",
    "Cats are independent animals that make great companions",
    "JavaScript is used for web development and runs in browsers",
    "Deep learning uses multi-layered neural networks",
    "Puppies require training and socialization from an early age"
]

print(f"Knowledge base: {len(documents)} documents")

Knowledge base: 8 documents


In [19]:
# Generate embeddings for all documents
print("Generating embeddings for all documents...")
doc_embeddings = model.encode(documents)

print(f"✅ Created {len(doc_embeddings)} embeddings")
print(f"Each embedding has {doc_embeddings[0].shape[0]} dimensions")

Generating embeddings for all documents...
✅ Created 8 embeddings
Each embedding has 384 dimensions


In [20]:
def search(query, documents, doc_embeddings, top_k=3):
    """
    Search for documents similar to the query.
    
    Args:
        query: Search query (string)
        documents: List of document texts
        doc_embeddings: Pre-computed document embeddings
        top_k: Number of results to return
    
    Returns:
        List of (document, similarity_score) tuples
    """
    # Embed the query
    query_embedding = model.encode(query)
    
    # Calculate similarities
    similarities = []
    for i, doc_emb in enumerate(doc_embeddings):
        similarity = cosine_similarity(query_embedding, doc_emb)
        similarities.append((documents[i], similarity))
    
    # Sort by similarity (highest first)
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    # Return top k results
    return similarities[:top_k]

print("✅ Search function ready!")

✅ Search function ready!


In [21]:
# Test different queries
queries = [
    "What is artificial intelligence?",
    "Tell me about pet dogs",
    "How do I code in Python?"
]

for query in queries:
    print(f"\n{'='*80}")
    print(f"QUERY: {query}")
    print(f"{'='*80}")
    
    results = search(query, documents, doc_embeddings, top_k=3)
    
    for i, (doc, score) in enumerate(results, 1):
        print(f"\n{i}. (Score: {score:.3f})")
        print(f"   {doc}")


QUERY: What is artificial intelligence?

1. (Score: 0.408)
   Machine learning enables computers to learn from data

2. (Score: 0.395)
   Neural networks are inspired by biological brains

3. (Score: 0.326)
   Python is a high-level programming language known for simplicity

QUERY: Tell me about pet dogs

1. (Score: 0.548)
   Dogs are loyal and friendly pets that need exercise

2. (Score: 0.437)
   Puppies require training and socialization from an early age

3. (Score: 0.413)
   Cats are independent animals that make great companions

QUERY: How do I code in Python?

1. (Score: 0.554)
   Python is a high-level programming language known for simplicity

2. (Score: 0.148)
   Puppies require training and socialization from an early age

3. (Score: 0.138)
   JavaScript is used for web development and runs in browsers


In [22]:
# Load two different models for comparison
print("Loading models...\n")

model_small = SentenceTransformer('all-MiniLM-L6-v2')      # 384 dimensions
model_large = SentenceTransformer('all-mpnet-base-v2')     # 768 dimensions

print("✅ Both models loaded!")
print(f"Small model: {model_small.get_sentence_embedding_dimension()} dimensions")
print(f"Large model: {model_large.get_sentence_embedding_dimension()} dimensions")

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000164D0B8A210>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: c30b1f74-5e92-4fcb-92b7-907917f74733)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Loading models...



'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000164D0B89F90>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 0583e410-2e6f-4ea4-b0a4-33f37f608c02)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000164D0B89A90>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 163153a5-ef29-420e-8f7d-b4f13d3682f2)')' thrown while requesting HEAD https://huggingface.

✅ Both models loaded!
Small model: 384 dimensions
Large model: 768 dimensions


In [23]:
# Compare on a similarity task
test_pairs = [
    ("The dog is running", "A canine is jogging"),           # Similar
    ("I love pizza", "Pizza is delicious"),                  # Related
    ("Python programming", "Cooking pasta")                  # Unrelated
]

print("Comparing model performance:\n")
for text1, text2 in test_pairs:
    # Small model
    emb1_small = model_small.encode([text1, text2])
    sim_small = cosine_similarity(emb1_small[0], emb1_small[1])
    
    # Large model  
    emb1_large = model_large.encode([text1, text2])
    sim_large = cosine_similarity(emb1_large[0], emb1_large[1])
    
    print(f"Pair: '{text1}' vs '{text2}'")
    print(f"  Small model: {sim_small:.3f}")
    print(f"  Large model: {sim_large:.3f}")
    print()

Comparing model performance:

Pair: 'The dog is running' vs 'A canine is jogging'
  Small model: 0.818
  Large model: 0.827

Pair: 'I love pizza' vs 'Pizza is delicious'
  Small model: 0.801
  Large model: 0.785

Pair: 'Python programming' vs 'Cooking pasta'
  Small model: 0.142
  Large model: 0.120



In [24]:
class SimpleRetriever:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        """
        Initialize retriever with embedding model.
        """
        self.model = SentenceTransformer(model_name)
        self.chunks = []
        self.embeddings = None
    
    def add_documents(self, documents, chunk_size=500):
        """
        Add documents to the retriever (chunks and embeds them).
        """
        # Simple chunking (from Module 2)
        for doc in documents:
            words = doc.split()
            for i in range(0, len(words), chunk_size):
                chunk = ' '.join(words[i:i+chunk_size])
                self.chunks.append(chunk)
        
        # Generate embeddings
        print(f"Embedding {len(self.chunks)} chunks...")
        self.embeddings = self.model.encode(self.chunks)
        print(f"✅ Ready! {len(self.chunks)} chunks indexed.")
    
    def search(self, query, top_k=3):
        """
        Search for relevant chunks.
        """
        # Embed query
        query_embedding = self.model.encode(query)
        
        # Calculate similarities
        similarities = []
        for i, chunk_emb in enumerate(self.embeddings):
            sim = cosine_similarity(query_embedding, chunk_emb)
            similarities.append((self.chunks[i], sim))
        
        # Sort and return top k
        similarities.sort(key=lambda x: x[1], reverse=True)
        return similarities[:top_k]

print("✅ SimpleRetriever class ready!")

✅ SimpleRetriever class ready!


In [25]:
# Test it with sample documents
sample_docs = [
    """
    Python is a versatile programming language widely used in web development,
    data science, and automation. Its simple syntax makes it beginner-friendly
    while remaining powerful for advanced applications.
    """,
    """
    Machine learning is a subset of artificial intelligence that enables systems
    to learn and improve from experience. Popular frameworks include TensorFlow,
    PyTorch, and scikit-learn.
    """,
    """
    Dogs are loyal companions that require regular exercise, training, and
    veterinary care. Different breeds have varying needs and temperaments.
    """
]

# Create retriever and add documents
retriever = SimpleRetriever()
retriever.add_documents(sample_docs, chunk_size=100)

# Test searches
test_queries = [
    "How do I start learning to code?",
    "What is AI and machine learning?",
    "Tell me about caring for pets"
]

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    results = retriever.search(query, top_k=2)
    for i, (chunk, score) in enumerate(results, 1):
        print(f"\nResult {i} (Score: {score:.3f}):")
        print(chunk.strip())

Embedding 3 chunks...
✅ Ready! 3 chunks indexed.

Query: How do I start learning to code?

Result 1 (Score: 0.263):
Python is a versatile programming language widely used in web development, data science, and automation. Its simple syntax makes it beginner-friendly while remaining powerful for advanced applications.

Result 2 (Score: 0.228):
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience. Popular frameworks include TensorFlow, PyTorch, and scikit-learn.

Query: What is AI and machine learning?

Result 1 (Score: 0.697):
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience. Popular frameworks include TensorFlow, PyTorch, and scikit-learn.

Result 2 (Score: 0.234):
Python is a versatile programming language widely used in web development, data science, and automation. Its simple syntax makes it beginner-friendly while remaining powerful for advanced applications.

In [26]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# List of sample sentences
sentences = [
    "The dog is playing in the park",
    "A puppy is running outside",
    "The cat is sleeping on the couch",
    "Python is a programming language",
    "Machine learning models need data",
    "I love coding in Python"
]

# Load the embedding model (all-MiniLM-L6-v2)
# This is a lightweight, high-performance sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all sentences
# Shape: (6, 384) - 384-dimensional vectors
embeddings = model.encode(sentences, normalize_embeddings=True)  # Normalize for cosine similarity

# Compute pairwise cosine similarities
# Result is a 6x6 matrix where similarity[i][j] is between sentence i and j
similarity_matrix = cosine_similarity(embeddings)

# Function to display similarities for a specific query sentence
def show_similarities(query_index: int, sentences: list, matrix: np.ndarray):
    print(f"Query: \"{sentences[query_index]}\"")
    print("Similarity scores:")
    # Sort by similarity descending (exclude self-similarity of 1.0)
    scores = [(i, matrix[query_index][i]) for i in range(len(sentences)) if i != query_index]
    scores.sort(key=lambda x: x[1], reverse=True)
    
    for idx, sim in scores:
        print(f"  {sim:.4f} → \"{sentences[idx]}\"")
    print()

# === Calculate and display results ===

print("=== Semantic Similarity Analysis using all-MiniLM-L6-v2 ===\n")

# Sentence 1 (index 0): "The dog is playing in the park"
show_similarities(0, sentences, similarity_matrix)

# Sentence 4 (index 3): "Python is a programming language"
show_similarities(3, sentences, similarity_matrix)

# Optional: Print full similarity matrix (rounded)
print("Full Cosine Similarity Matrix:")
print(np.round(similarity_matrix, 4))

=== Semantic Similarity Analysis using all-MiniLM-L6-v2 ===

Query: "The dog is playing in the park"
Similarity scores:
  0.3984 → "A puppy is running outside"
  0.0987 → "Python is a programming language"
  0.0902 → "I love coding in Python"
  0.0714 → "The cat is sleeping on the couch"
  -0.0052 → "Machine learning models need data"

Query: "Python is a programming language"
Similarity scores:
  0.7304 → "I love coding in Python"
  0.1133 → "Machine learning models need data"
  0.0987 → "The dog is playing in the park"
  0.0395 → "A puppy is running outside"
  0.0199 → "The cat is sleeping on the couch"

Full Cosine Similarity Matrix:
[[ 1.      0.3984  0.0714  0.0987 -0.0052  0.0902]
 [ 0.3984  1.      0.0404  0.0395  0.0418  0.0016]
 [ 0.0714  0.0404  1.      0.0199 -0.0485  0.0474]
 [ 0.0987  0.0395  0.0199  1.      0.1133  0.7304]
 [-0.0052  0.0418 -0.0485  0.1133  1.      0.1116]
 [ 0.0902  0.0016  0.0474  0.7304  0.1116  1.    ]]


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# Helper function for mean pooling
def mean_pooling(token_embeddings, attention_mask):
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Load model and tokenizer
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

document = """
Artificial intelligence (AI) is intelligence demonstrated by machines, in contrast to
the natural intelligence displayed by humans and animals. Leading AI textbooks define
the field as the study of intelligent agents: any device that perceives its environment
and takes actions that maximize its chance of successfully achieving its goals.
Machine learning is a subset of artificial intelligence that focuses on the use of data
and algorithms to imitate the way that humans learn, gradually improving its accuracy.
Machine learning is an important component of the growing field of data science.
Deep learning is part of a broader family of machine learning methods based on artificial
neural networks with representation learning. Learning can be supervised, semi-supervised
or unsupervised. Deep learning architectures such as deep neural networks, deep belief
networks, recurrent neural networks and convolutional neural networks have been applied
to fields including computer vision, speech recognition, natural language processing,
machine translation, and bioinformatics.
Natural language processing is a subfield of linguistics, computer science, and artificial
intelligence concerned with the interactions between computers and human language, in
particular how to program computers to process and analyze large amounts of natural
language data. Challenges in natural language processing frequently involve speech
recognition, natural language understanding, and natural language generation.
"""

query = "What is machine learning?"

# Encode query
query_inputs = tokenizer(query, return_tensors='pt', truncation=False)
with torch.no_grad():
    query_outputs = model(**query_inputs)
query_emb = mean_pooling(query_outputs.last_hidden_state, query_inputs['attention_mask'])
query_emb = F.normalize(query_emb, p=2, dim=1)[0].cpu().numpy()

def chunk_document(text: str, max_chars: int):
    text = text.strip()
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            last_space = text.rfind(' ', start, end)
            if last_space != -1:
                end = last_space
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end + (1 if end < len(text) and text[end] == ' ' else 0)
    return chunks

sizes = {'Small': 100, 'Medium': 200, 'Large': 400}

print("Chunk Size Comparison:\n")

for name, size in sizes.items():
    chunks = chunk_document(document, size)
    num_chunks = len(chunks)
    
    if num_chunks == 0:
        continue
    
    # Encode all chunks
    chunk_inputs = tokenizer(chunks, padding=True, truncation=False, return_tensors='pt')
    with torch.no_grad():
        chunk_outputs = model(**chunk_inputs)
    chunk_embs = mean_pooling(chunk_outputs.last_hidden_state, chunk_inputs['attention_mask'])
    chunk_embs = F.normalize(chunk_embs, p=2, dim=1).cpu().numpy()
    
    # Compute similarities
    sims = np.dot(chunk_embs, query_emb)
    
    # Top 3
    top_indices = np.argsort(sims)[-3:][::-1]
    
    print(f"{name} Chunks ({size} chars):")
    print(f"- Number of chunks: {num_chunks}")
    print(f"- Top result: \"{chunks[top_indices[0]].replace('\n', ' ')}\"")
    print(f"- Score: {sims[top_indices[0]]:.4f}")
    print("- Analysis: ", end="")
    
    if name == "Small":
        print("Very focused on the exact definition sentence. High precision but may miss broader context (e.g., relation to data science).")
    elif name == "Medium":
        print("Good balance: captures the core definition and some additional context without irrelevant information.")
    elif name == "Large":
        print("More complete context but includes unrelated topics (e.g., deep learning details or NLP), reducing focus.")
    print()

print("Best chunk size for this use case: Medium (200 characters) because it provides the most precise and relevant chunk as the top result while including useful surrounding context, achieving the best balance between focus and completeness.")